<a href="https://colab.research.google.com/github/RyanKhurana9/Pytorch/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [150]:
import pandas as pd


In [151]:
df=pd.read_csv("100_Unique_QA_Dataset.csv")

In [152]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [153]:
df.shape[0] # total number of rows


90

In [154]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace("?",'')
  text=text.replace("'",'')

  return text.split()


In [155]:
tokenize("Who wrote 'To Kill a Mockingbird'?")

['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']

In [156]:
# vocab(unique word in dataset)
vocab={"<UNK>":0}#<UNK>->unknown

In [157]:
len(vocab)

1

In [158]:
def build_vocab(row):
  tokenize_question=tokenize(row['question'])
  tokenize_answer=tokenize(row['answer'])
  # merged_token
  merged_token=tokenize_question+tokenize_answer
  for word in merged_token:
    if word not in vocab:
      vocab[word]=len(vocab)
    else:
      continue
  return vocab



In [159]:
df.apply(build_vocab,axis=1)

,0
0,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
1,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
2,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
3,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
4,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
...,...
85,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
86,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
87,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."
88,"{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'ca..."


In [160]:
len(vocab)

324

In [161]:
tokenize("What is the capital of India")

['what', 'is', 'the', 'capital', 'of', 'india']

In [162]:
# convert word to numerical_indicies
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append("<UNK>")
  return indexed_text



In [163]:
text_to_indices("What is the capital of India",vocab)

[1, 2, 3, 4, 5, 73]

In [164]:
import torch
from torch.utils.data import Dataset,DataLoader

In [165]:
class QAdataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab
  def __len__(self):
    return df.shape[0]
  def __getitem__(self, index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)


In [166]:
dataset=QAdataset(df,vocab)

In [167]:
len(dataset)

90

In [168]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [169]:
dataset[10
        ]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [170]:
df.iloc[10]

,10
question,What is the capital of Japan?
answer,Tokyo


In [171]:
dataloader=DataLoader(dataset,batch_size=1
                      ,shuffle=True)

In [172]:
len(dataloader)

90

In [173]:
for question ,answer in dataloader:
  print("Question:",question)
  print("answer:",answer[0])
  print("\n")

Question: tensor([[  1,   2,   3,   4,   5, 206]])
answer: tensor([207])


Question: tensor([[  1,   2,   3, 234,   5, 235]])
answer: tensor([131])


Question: tensor([[  1,   2,   3, 180, 181, 182, 183]])
answer: tensor([184])


Question: tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]])
answer: tensor([155])


Question: tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]])
answer: tensor([205])


Question: tensor([[ 10,  75,   3, 296,  19, 297]])
answer: tensor([298])


Question: tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]])
answer: tensor([220])


Question: tensor([[42, 18,  2, 62, 63,  3, 64, 18]])
answer: tensor([65])


Question: tensor([[ 42,  18, 118,   3, 186, 187]])
answer: tensor([188])


Question: tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]])
answer: tensor([173])


Question: tensor([[1, 2, 3, 4, 5, 8]])
answer: tensor([9])


Question: tensor([[ 1,  2,  3, 69,  5, 53]])
answer: tensor([260])


Question: tensor([[  1,   2,   3,   4,   5, 236, 237]])
answer: 

All sequences in a batch must have equal lengths for PyTorch's default DataLoader collation to stack them into a single tensor.

In [174]:
import torch.nn as nn

In [175]:
class RNNmodel(nn.Module):
  def __init__(self, vocab_size):
     super().__init__()
     # super -> refers to the parent class
     # it allows the child class to invoke the constructor of its parent class
     self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
     self.rnn = nn.RNN(50, 64, batch_first=True)
     self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    # self.rnn returns (output, hidden)
    output, final = self.rnn(embedded_question)
    # final has shape [1, batch_size, hidden_dim]. We squeeze it to [batch_size, hidden_dim]
    output = self.fc(final.squeeze(0))
    return output

In [176]:
len(vocab)

324

In [177]:
x=nn.Embedding(324,embedding_dim=50)
y=nn.RNN(50,64,batch_first=True)
z=nn.Linear(64,324)# The linear layer returns a vector of 324 scores (logits), one for each token in the vocabulary(# find the max)



In [178]:
dataset[0][0].shape# question

torch.Size([6])

In [179]:
dataset[0][0].reshape(1,6)

tensor([[1, 2, 3, 4, 5, 6]])

In [180]:
a=dataset[0][0].reshape(1,6)
print("shape of a:",a.shape)
# send this to embedding
b=x(a)
print("shape of b:",b.shape)
# send this to rnn
c,d=y(b)
print("shape of c:",c.shape)# sub hidden states
print("shape of d:",d.shape)# final hidden state
# send this to linear

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])


In [181]:
model = RNNmodel(len(vocab))

In [182]:
learning_rate=0.01
epochs=50


In [183]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [184]:
for epoch in range(epochs):
  total_loss=0
  for question,answer in dataloader:
    optimizer.zero_grad()
    output=model(question)
    loss=criterion(output,answer[0])
    # backprop to calculate gradients
    loss.backward()
    # update weights
    optimizer.step()
    total_loss+=loss.item()
  print("epoch:",epoch+1,"loss:",total_loss/len(dataloader))

epoch: 1 loss: 6.0423002666897245
epoch: 2 loss: 3.859004282951355
epoch: 3 loss: 1.7547657060126463
epoch: 4 loss: 0.6119910633398427
epoch: 5 loss: 0.4447894969592906
epoch: 6 loss: 0.3422469241699825
epoch: 7 loss: 0.29166377517653425
epoch: 8 loss: 0.16668761942095847
epoch: 9 loss: 0.28066480747850925
epoch: 10 loss: 0.1816278096404858
epoch: 11 loss: 0.11219884771594985
epoch: 12 loss: 0.1488806101364187
epoch: 13 loss: 0.23753853436404218
epoch: 14 loss: 0.13482198094328243
epoch: 15 loss: 0.1747058130587296
epoch: 16 loss: 0.1142806402388184
epoch: 17 loss: 0.10017812601290643
epoch: 18 loss: 0.04390287235405089
epoch: 19 loss: 0.04089229017424966
epoch: 20 loss: 0.03491816056969886
epoch: 21 loss: 0.039556246073657854
epoch: 22 loss: 0.018936046437334476
epoch: 23 loss: 0.0038381793471570645
epoch: 24 loss: 0.003209733710779498
epoch: 25 loss: 0.0028234598285052925
epoch: 26 loss: 0.002540066112608959
epoch: 27 loss: 0.0023143935431208873
epoch: 28 loss: 0.002131012239260599
e

logits = z(last_output)  # shape: (1, 324)probabilities = torch.softmax(logits, dim=1)  # shape: (1, 324)



- (1, 324) = 1 sample with 324 vocabulary scores.
- Softmax converts those 324 scores into probabilities.
- The output remains (1, 324), but the values now sum to 1.
- Argmax selects the token with the highest probability.

In [199]:
def predict(model,question,threshold=0.5):
  numerical_question=text_to_indices(question,vocab)
  print(numerical_question)
  # convert the list into a tesor
  numerical_question=torch.tensor(numerical_question)
  numerical_question=numerical_question.unsqueeze(0)
  print(numerical_question.shape)
  # send the numerical question to the model
  output=model(numerical_question)
  prob=torch.nn.functional.softmax(output,dim=1)
  val,indx=torch.max(prob,dim=1)
  if(val<threshold):
    print("I dont know")
  else:
    for key,value in vocab.items():
      if(value==indx):
        print(key)
        break



In [200]:
predict(model, "What is the largest planet in our solar system?")

[1, 2, 3, 17, 18, 19, 20, 21, 22]
torch.Size([1, 9])
jupiter
